# Notebook 2 – Train Models

**Purpose:** Download the prepared dataset from S3, train four families of models for ASL alphabet recognition, and upload all trained model files back to S3.

**Models trained:**
| # | Approach | Architectures |
|---|----------|--------------|
| 1 | Hand Landmark MLP (63 MediaPipe features → MLP) | Custom MLP |
| 2 | CNN on Raw Images | MobileNetV2, EfficientNetB0, ResNet50 |
| 3 | CNN on Cropped Hand Images | MobileNetV2, EfficientNetB0, ResNet50 |
| 4 | CNN on Hand Skeleton Images | MobileNetV2, EfficientNetB0, ResNet50 |

Each CNN uses a frozen-backbone phase followed by fine-tuning of the top 20 layers.  
All pipelines use `EarlyStopping(patience=5)`, `ModelCheckpoint`, and `ReduceLROnPlateau`.

**Prerequisites:** Run `01_setup_and_data.ipynb` first.

## Step 1 – Load config.json & Imports

In [ ]:
import json, os, sys, boto3
import numpy as np

# ── Load config written by Notebook 1 ─────────────────────────────────────────
try:
    with open("config.json") as f:
        cfg = json.load(f)
    BUCKET       = cfg["bucket"]
    PREFIX       = cfg["prefix"]
    CLASS_NAMES  = cfg["class_names"]
    NUM_CLASSES  = len(CLASS_NAMES)
    print(f"✅ Config loaded — bucket: {BUCKET}, classes: {NUM_CLASSES}")
except FileNotFoundError:
    print("❌ config.json not found. Run Notebook 1 first.")
    raise

In [ ]:
import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ {len(gpus)} GPU(s) available: {[g.name for g in gpus]}")
else:
    print("⚠️  No GPU detected — training on CPU will be slow. Consider switching to ml.g4dn.xlarge.")

## Step 2 – Download Dataset from S3

In [ ]:
import boto3, os
from pathlib import Path

s3 = boto3.client("s3")

DOWNLOAD_DIR = "/tmp/asl_split"

def download_s3_prefix(bucket, s3_prefix, local_dir):
    """Download all objects under s3_prefix to local_dir."""
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=s3_prefix)
    keys = [obj["Key"] for page in pages for obj in page.get("Contents", [])]
    print(f"Downloading {len(keys)} files from s3://{bucket}/{s3_prefix} …")
    for i, key in enumerate(keys):
        rel  = key[len(s3_prefix):].lstrip("/")
        dest = os.path.join(local_dir, rel)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if not os.path.exists(dest):          # skip already downloaded
            s3.download_file(bucket, key, dest)
        if (i + 1) % 5000 == 0:
            print(f"  … {i+1}/{len(keys)}")
    print(f"  ✅ Done → {local_dir}")

for split in ["train", "val", "test"]:
    local = os.path.join(DOWNLOAD_DIR, split)
    if os.path.exists(local) and len(list(Path(local).rglob("*.jpg"))) > 100:
        print(f"  ⏩ {split}: already cached at {local}")
    else:
        download_s3_prefix(BUCKET, f"{PREFIX}/{split}", local)

TRAIN_DIR = os.path.join(DOWNLOAD_DIR, "train")
VAL_DIR   = os.path.join(DOWNLOAD_DIR, "val")
TEST_DIR  = os.path.join(DOWNLOAD_DIR, "test")
print("\nAll splits ready.")

## Step 3 – Shared Training Helpers

In [ ]:
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 32
MAX_EPOCHS   = 50
FT_EPOCHS    = 20
LR_BASE      = 1e-4
LR_FINE_TUNE = 1e-5
PATIENCE     = 5
SEED         = 42
MODELS_DIR   = "/tmp/asl_models"
os.makedirs(MODELS_DIR, exist_ok=True)

def set_seeds(seed=SEED):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def get_callbacks(name):
    ckpt_path = os.path.join(MODELS_DIR, f"{name}_best.keras")
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=PATIENCE, mode="max",
            restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(
            filepath=ckpt_path, monitor="val_accuracy",
            save_best_only=True, mode="max", verbose=1),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_accuracy", factor=0.5, patience=max(2, PATIENCE//2),
            min_lr=1e-7, verbose=1),
    ]

def make_generators(train_dir, val_dir, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE, seed=SEED):
    train_gen = ImageDataGenerator(
        rescale=1./255, rotation_range=15, width_shift_range=0.10,
        height_shift_range=0.10, zoom_range=0.10, brightness_range=[0.8, 1.2],
        fill_mode="nearest")
    val_gen = ImageDataGenerator(rescale=1./255)
    train_flow = train_gen.flow_from_directory(
        train_dir, target_size=target_size, batch_size=batch_size,
        class_mode="categorical", shuffle=True, seed=seed)
    val_flow = val_gen.flow_from_directory(
        val_dir, target_size=target_size, batch_size=batch_size,
        class_mode="categorical", shuffle=False)
    return train_flow, val_flow

print("✅ Helpers ready.")

## Step 4 – Approach 1: Hand Landmark MLP

In [ ]:
import mediapipe as mp
import cv2

print("MediaPipe version:", mp.__version__)

def extract_landmarks_batch(paths, labels, augment=False, aug_reps=3, seed=SEED):
    """Extract 63 MediaPipe landmark features from a list of images."""
    from tensorflow.keras.utils import to_categorical
    label_map = {c: i for i, c in enumerate(CLASS_NAMES)}
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1,
                           min_detection_confidence=0.7)
    rng = np.random.default_rng(seed)
    X, y = [], []
    for path, lbl in zip(paths, labels):
        img = cv2.imread(path)
        if img is None:
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)
        if not result.multi_hand_landmarks:
            continue
        feats = np.array([[lm.x, lm.y, lm.z] for lm in result.multi_hand_landmarks[0].landmark],
                         dtype=np.float32).flatten()
        X.append(feats); y.append(label_map.get(lbl, 0))
        if augment:
            for rep in range(aug_reps):
                noise = rng.normal(0, 0.01, feats.shape)
                scale = rng.uniform(0.95, 1.05)
                X.append((feats + noise) * scale)
                y.append(label_map.get(lbl, 0))
    hands.close()
    return (np.array(X, dtype=np.float32),
            to_categorical(np.array(y, dtype=np.int32), num_classes=NUM_CLASSES))

def collect_paths_labels(directory):
    paths, labels = [], []
    for cls in sorted(os.listdir(directory)):
        cls_dir = os.path.join(directory, cls)
        if not os.path.isdir(cls_dir): continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(cls_dir, fname))
                labels.append(cls)
    return paths, labels

print("Collecting training paths …")
tr_paths, tr_labels = collect_paths_labels(TRAIN_DIR)
vl_paths, vl_labels = collect_paths_labels(VAL_DIR)
print(f"Train images: {len(tr_paths)}, Val images: {len(vl_paths)}")

In [ ]:
print("Extracting training landmarks (this takes ~15–30 min without GPU) …")
try:
    X_train_lm, y_train_lm = extract_landmarks_batch(tr_paths, tr_labels, augment=True)
    X_val_lm,   y_val_lm   = extract_landmarks_batch(vl_paths, vl_labels, augment=False)
    print(f"✅ X_train: {X_train_lm.shape}, X_val: {X_val_lm.shape}")
except MemoryError:
    print("❌ OOM — reduce augmentation reps or use a larger instance.")
    raise

In [ ]:
set_seeds()

def build_landmark_mlp(input_dim=63, num_classes=NUM_CLASSES):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(num_classes, activation="softmax"),
    ], name="landmark_mlp")
    model.compile(optimizer=keras.optimizers.Adam(LR_BASE),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    return model

mlp_model = build_landmark_mlp()
mlp_model.summary()

In [ ]:
mlp_history = mlp_model.fit(
    X_train_lm, y_train_lm,
    validation_data=(X_val_lm, y_val_lm),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks("landmark_mlp"), verbose=1)

mlp_path = os.path.join(MODELS_DIR, "landmark_mlp.keras")
mlp_model.save(mlp_path)
best_val_mlp = max(mlp_history.history["val_accuracy"])
print(f"✅ Landmark MLP saved. Best val_accuracy: {best_val_mlp:.4f}")

## Step 5 – Approach 2/3/4: CNN Models (Raw, Cropped, Skeleton)

In [ ]:
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, ResNet50

BACKBONE_MAP = {
    "MobileNetV2":  MobileNetV2,
    "EfficientNetB0": EfficientNetB0,
    "ResNet50":     ResNet50,
}

def build_cnn(architecture, input_shape=(224,224,3), num_classes=NUM_CLASSES,
              dense_units=256, dropout=0.5, lr=LR_BASE, trainable_base=False):
    base_cls = BACKBONE_MAP[architecture]
    base = base_cls(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = trainable_base
    inputs  = keras.Input(shape=input_shape)
    x       = base(inputs, training=False)
    x       = keras.layers.GlobalAveragePooling2D()(x)
    x       = keras.layers.Dense(dense_units, activation="relu")(x)
    x       = keras.layers.Dropout(dropout)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    model   = keras.Model(inputs, outputs, name=f"cnn_{architecture.lower()}")
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    return model, base

def unfreeze_top(model, base, num_layers=20, lr=LR_FINE_TUNE):
    base.trainable = True
    for layer in base.layers[:-num_layers]:
        layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    return model

print("✅ CNN builder ready.")

In [ ]:
def train_cnn_approach(approach_name, train_data_dir, val_data_dir,
                       architecture="MobileNetV2"):
    """Train one CNN for one approach + architecture combination."""
    model_name = f"cnn_{approach_name}_{architecture.lower()}"
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"{'='*60}")
    
    set_seeds()
    train_flow, val_flow = make_generators(train_data_dir, val_data_dir)
    model, base = build_cnn(architecture)
    
    # Phase 1 – frozen backbone
    history = model.fit(train_flow, validation_data=val_flow,
                        epochs=MAX_EPOCHS, callbacks=get_callbacks(model_name + "_p1"),
                        verbose=1)
    
    # Phase 2 – fine-tune top 20 layers
    model = unfreeze_top(model, base)
    history_ft = model.fit(train_flow, validation_data=val_flow,
                           epochs=FT_EPOCHS, callbacks=get_callbacks(model_name + "_ft"),
                           verbose=1)
    
    best_val = max(history.history["val_accuracy"] + history_ft.history["val_accuracy"])
    
    save_path = os.path.join(MODELS_DIR, f"{model_name}.keras")
    model.save(save_path)
    print(f"✅ Saved: {save_path}  (best val_accuracy: {best_val:.4f})")
    return {"model_name": model_name, "save_path": save_path, "best_val_accuracy": float(best_val)}

all_results = []

In [ ]:
# ── Approach 2: CNN on Raw Images ─────────────────────────────────────────────
for arch in ["MobileNetV2", "EfficientNetB0", "ResNet50"]:
    try:
        res = train_cnn_approach("raw", TRAIN_DIR, VAL_DIR, architecture=arch)
        all_results.append(res)
    except Exception as e:
        print(f"❌ cnn_raw_{arch} failed: {e}")

In [ ]:
# ── Approach 3: CNN on Cropped Hand Images ─────────────────────────────────────
# Pre-generate cropped images using MediaPipe bounding boxes
import cv2, mediapipe as mp, shutil

CROPPED_TRAIN = "/tmp/asl_cropped/train"
CROPPED_VAL   = "/tmp/asl_cropped/val"

def crop_and_save(src_dir, dst_dir, padding=0.20):
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.7)
    saved = failed = 0
    for cls in sorted(os.listdir(src_dir)):
        cls_src = os.path.join(src_dir, cls)
        if not os.path.isdir(cls_src): continue
        cls_dst = os.path.join(dst_dir, cls)
        os.makedirs(cls_dst, exist_ok=True)
        for fname in os.listdir(cls_src):
            if not fname.lower().endswith((".jpg",".jpeg",".png")): continue
            src_path = os.path.join(cls_src, fname)
            dst_path = os.path.join(cls_dst, fname)
            if os.path.exists(dst_path):
                saved += 1; continue
            img = cv2.imread(src_path)
            if img is None: failed += 1; continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            result = hands.process(rgb)
            if not result.multi_hand_landmarks:
                shutil.copy2(src_path, dst_path)   # fallback: use raw image
                saved += 1; continue
            lms = result.multi_hand_landmarks[0].landmark
            h, w = img.shape[:2]
            xs = [lm.x for lm in lms]; ys = [lm.y for lm in lms]
            pad_x = padding * (max(xs) - min(xs)); pad_y = padding * (max(ys) - min(ys))
            x1 = max(0, int((min(xs)-pad_x)*w)); y1 = max(0, int((min(ys)-pad_y)*h))
            x2 = min(w, int((max(xs)+pad_x)*w)); y2 = min(h, int((max(ys)+pad_y)*h))
            crop = img[y1:y2, x1:x2]
            if crop.size > 0:
                cv2.imwrite(dst_path, crop); saved += 1
            else:
                shutil.copy2(src_path, dst_path); saved += 1
    hands.close()
    print(f"  Cropped: {saved} saved, {failed} failed → {dst_dir}")

if not os.path.exists(CROPPED_TRAIN) or len(list(os.scandir(CROPPED_TRAIN))) == 0:
    print("Cropping training images …")
    crop_and_save(TRAIN_DIR, CROPPED_TRAIN)
if not os.path.exists(CROPPED_VAL) or len(list(os.scandir(CROPPED_VAL))) == 0:
    print("Cropping validation images …")
    crop_and_save(VAL_DIR, CROPPED_VAL)
print("✅ Cropped images ready.")

In [ ]:
for arch in ["MobileNetV2", "EfficientNetB0", "ResNet50"]:
    try:
        res = train_cnn_approach("cropped", CROPPED_TRAIN, CROPPED_VAL, architecture=arch)
        all_results.append(res)
    except Exception as e:
        print(f"❌ cnn_cropped_{arch} failed: {e}")

In [ ]:
# ── Approach 4: CNN on Skeleton Images ─────────────────────────────────────────
SKEL_TRAIN = "/tmp/asl_skeleton/train"
SKEL_VAL   = "/tmp/asl_skeleton/val"
BG_COLOR   = (255, 255, 255)
CONN_COLOR = (0, 0, 0)
DOT_COLOR  = (0, 0, 255)

def generate_skeletons(src_dir, dst_dir, output_size=(224,224)):
    import cv2, mediapipe as mp
    mp_hands  = mp.solutions.hands
    hands     = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.7)
    saved = failed = 0
    h_out, w_out = output_size
    for cls in sorted(os.listdir(src_dir)):
        cls_src = os.path.join(src_dir, cls)
        if not os.path.isdir(cls_src): continue
        cls_dst = os.path.join(dst_dir, cls)
        os.makedirs(cls_dst, exist_ok=True)
        for fname in os.listdir(cls_src):
            if not fname.lower().endswith((".jpg",".jpeg",".png")): continue
            src_path = os.path.join(cls_src, fname)
            dst_path = os.path.join(cls_dst, fname)
            if os.path.exists(dst_path):
                saved += 1; continue
            img = cv2.imread(src_path)
            if img is None: failed += 1; continue
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            result = hands.process(rgb)
            if not result.multi_hand_landmarks:
                failed += 1; continue
            canvas = np.full((h_out, w_out, 3), BG_COLOR, dtype=np.uint8)
            lms = result.multi_hand_landmarks[0].landmark
            for conn in mp_hands.HAND_CONNECTIONS:
                s, e = conn
                p1 = (int(lms[s].x*w_out), int(lms[s].y*h_out))
                p2 = (int(lms[e].x*w_out), int(lms[e].y*h_out))
                cv2.line(canvas, p1, p2, CONN_COLOR, 2)
            for lm in lms:
                cv2.circle(canvas, (int(lm.x*w_out), int(lm.y*h_out)), 4, DOT_COLOR, -1)
            cv2.imwrite(dst_path, canvas)
            saved += 1
    hands.close()
    print(f"  Skeletons: {saved} saved, {failed} failed → {dst_dir}")

if not os.path.exists(SKEL_TRAIN) or len(list(os.scandir(SKEL_TRAIN))) == 0:
    print("Generating training skeleton images …")
    generate_skeletons(TRAIN_DIR, SKEL_TRAIN)
if not os.path.exists(SKEL_VAL) or len(list(os.scandir(SKEL_VAL))) == 0:
    print("Generating validation skeleton images …")
    generate_skeletons(VAL_DIR, SKEL_VAL)
print("✅ Skeleton images ready.")

In [ ]:
for arch in ["MobileNetV2", "EfficientNetB0", "ResNet50"]:
    try:
        res = train_cnn_approach("skeleton", SKEL_TRAIN, SKEL_VAL, architecture=arch)
        all_results.append(res)
    except Exception as e:
        print(f"❌ cnn_skeleton_{arch} failed: {e}")

## Step 6 – Save Results Summary & Upload All Models to S3

In [ ]:
import json, boto3

s3 = boto3.client("s3")
MODELS_PREFIX = cfg["models_prefix"]

# Save local results
results_path = os.path.join(MODELS_DIR, "training_results.json")
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)
print("Training results summary:")
print(json.dumps(all_results, indent=2))

In [ ]:
# Upload all .keras model files to S3
import glob

model_files = glob.glob(os.path.join(MODELS_DIR, "*.keras"))
print(f"Uploading {len(model_files)} model files …")
for fpath in model_files:
    fname = os.path.basename(fpath)
    s3_key = f"{MODELS_PREFIX}/{fname}"
    try:
        s3.upload_file(fpath, BUCKET, s3_key)
        print(f"  ✅ s3://{BUCKET}/{s3_key}")
    except Exception as e:
        print(f"  ❌ Failed: {fname}: {e}")

# Upload results JSON
s3.upload_file(results_path, BUCKET, f"{MODELS_PREFIX}/training_results.json")
print("\n✅ All uploads complete.")

In [ ]:
# Update config.json with model URIs
model_uris = {}
for res in all_results:
    fname = os.path.basename(res["save_path"])
    model_uris[res["model_name"]] = f"s3://{BUCKET}/{MODELS_PREFIX}/{fname}"
# Add MLP
mlp_fname = os.path.basename(mlp_path)
model_uris["landmark_mlp"] = f"s3://{BUCKET}/{MODELS_PREFIX}/{mlp_fname}"

cfg["model_uris"] = model_uris
with open("config.json", "w") as f:
    json.dump(cfg, f, indent=2)
print("✅ config.json updated with model URIs.")
print(json.dumps(model_uris, indent=2))

---
## ✅ Notebook 2 Complete

All models have been trained and uploaded to:
```
s3://<BUCKET>/<PREFIX>/models/
    landmark_mlp.keras
    cnn_raw_mobilenetv2.keras
    cnn_raw_efficientnetb0.keras
    cnn_raw_resnet50.keras
    cnn_cropped_mobilenetv2.keras
    cnn_cropped_efficientnetb0.keras
    cnn_cropped_resnet50.keras
    cnn_skeleton_mobilenetv2.keras
    cnn_skeleton_efficientnetb0.keras
    cnn_skeleton_resnet50.keras
    training_results.json
```

**Next step → open `03_evaluate_and_select_best.ipynb`**